In [1]:
#%% stream rgb and depth (m)
import pyrealsense2 as rs
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
from matplotlib.figure import Figure
import io

# Setup
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)

# Start
pipeline.start(config)

# Align depth to color
align = rs.align(rs.stream.color)

# depth scale
depth_scale = pipeline.get_active_profile().get_device().first_depth_sensor().get_depth_scale()

print("Streaming aligned RGB and Depth. Press 'q' to quit.")

def create_depth_with_colorbar(depth, depth_scale, figsize=(6, 5)):
    """Create depth image with colormap and colorbar"""
    # Create figure
    fig = Figure(figsize=figsize, dpi=100)
    ax = fig.add_subplot(111)
    
    # Convert depth to meters for display
    depth_m = depth * depth_scale
    
    # Display with jet colormap
    im = ax.imshow(depth_m, cmap='jet', vmin=0, vmax=10)  # vmax=10 meters 
    ax.axis('off')
    
    # Add colorbar
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Depth (m)', fontsize=10)
    
    # Render to numpy array
    canvas = FigureCanvas(fig)
    canvas.draw()
    
    # Convert to numpy array
    buf = canvas.buffer_rgba()
    img = np.asarray(buf)
    
    plt.close(fig)
    return img

try:
    while True:
        # Get frames and align
        frames = pipeline.wait_for_frames()
        aligned_frames = align.process(frames)
        
        # Get aligned frames
        color_frame = aligned_frames.get_color_frame()
        depth_frame = aligned_frames.get_depth_frame()
        
        if not color_frame or not depth_frame:
            continue
        
        # Convert to numpy
        color = np.asanyarray(color_frame.get_data())
        depth = np.asanyarray(depth_frame.get_data())
        
        # Create depth with colorbar
        depth_with_cbar = create_depth_with_colorbar(depth, depth_scale)
        
        # Display
        cv2.imshow("RGB", color)
        cv2.imshow("Depth", depth_with_cbar)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()

RuntimeError: No device connected

In [ ]:
# stream and save rgb + depth recording, raw depth frames, raw rgb image, scaling factor

import pyrealsense2 as rs
import numpy as np
import cv2
import json
import os
from datetime import datetime

def create_timestamp_folder(base_path="./"):
    #timestamp folder. each acquisition is save as session_date_time  
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Main session folder
    session_folder = os.path.join(base_path, f"session_{timestamp}")
    
    # Subfolders
    folders = {
        'color_video': os.path.join(session_folder, "color_video"),
        'depth_video': os.path.join(session_folder, "depth_video"),
        'depth_scale_json': os.path.join(session_folder, "depth_scale_json"),
        'depth_raw': os.path.join(session_folder, "depth_raw"),
        'rgb_frames': os.path.join(session_folder, "rgb_frames"),
        
    }
    
    # Create folders
    for folder in folders.values():
        os.makedirs(folder, exist_ok=True)
    
    # Save folder structure information
    session_info = {
        'timestamp': timestamp,
        'session_folder': session_folder,
        'folders': folders
    }
    
    return session_info, folders

def record_video_and_raw():
    """Record RGB video, depth video, and raw depth files with ALIGNMENT"""
    
    # Create timestamped folder structure
    session_info, folders = create_timestamp_folder()
    
    # print("="*70)
    print("Recording")
    # print("="*70)
    print(f"Session: {session_info['timestamp']}")
    print(f"Folder: {session_info['session_folder']}")
    # print("="*70)
    
    # Setup camera
    pipeline = rs.pipeline()
    config = rs.config()
    
    # Enable streams
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
    
    # Start pipeline
    profile = pipeline.start(config)
    
    # Get depth scale
    depth_sensor = profile.get_device().first_depth_sensor()
    depth_scale = depth_sensor.get_depth_scale()
    
    
    # Align depth to color (or color to depth)
    align_to = rs.align(rs.stream.color) 
    
    # Save depth scale
    scale_file = os.path.join(folders['depth_scale_json'], "depth_scale.json")
    with open(scale_file, 'w') as f:
        json.dump({
            "depth_scale": depth_scale,
            "timestamp": session_info['timestamp'],
        }, f, indent=2)
    print(f"   Depth scale: {depth_scale}")
    
    # Video writers
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    color_video_path = os.path.join(folders['color_video'], "color_video.avi")
    depth_video_path = os.path.join(folders['depth_video'], "depth_video.avi")
    
    color_writer = cv2.VideoWriter(color_video_path, fourcc, 30, (640, 480))
    depth_writer = cv2.VideoWriter(depth_video_path, fourcc, 30, (640, 480))
    
    # Recording parameters
    frame_count = 0
    save_interval = 1  # Save every frame
    max_frames = 600  # 20 seconds at 30fps, if you want 1000 frames then change to 1000 --> at 30fps, the acquisiton will last 33 sec
    
    print(f"\nRecording {max_frames} frames...")
    print("Press 'q' to stop early")
    
    try:
        while frame_count < max_frames:
            # Wait for frames
            frames = pipeline.wait_for_frames()
            
            # ALIGN: Align depth to color
            aligned_frames = align_to.process(frames)
            
            # Get aligned frames
            color_frame = aligned_frames.get_color_frame()
            depth_frame = aligned_frames.get_depth_frame()
            
            if not color_frame or not depth_frame:
                continue
            
            # Convert to numpy
            color_bgr = np.asanyarray(color_frame.get_data())
            depth = np.asanyarray(depth_frame.get_data())  #unit16 raw depth information
            
            # Convert BGR to RGB for saving/opening later
            color_rgb = cv2.cvtColor(color_bgr, cv2.COLOR_BGR2RGB)
            
            # Write to videos (BGR format for video writer)
            color_writer.write(color_bgr)
            
            # Normalize depth for video visualization
            #depth_vis = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            #depth_writer.write(cv2.cvtColor(depth_vis, cv2.COLOR_GRAY2BGR))

            depth_norm = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
            depth_vis = cv2.applyColorMap(depth_norm, cv2.COLORMAP_JET)
            depth_writer.write(depth_vis)
            
            # Save raw depth and RGB frames - synchronized every frame
            if frame_count % save_interval == 0:
                # Save raw depth aligned/synchronized to RGB image)

                #save raw depth information variable depth as  .npy
                raw_file = os.path.join(folders['depth_raw'], f"depth_{frame_count:04d}_raw.npy")
                np.save(raw_file, depth)
                
                #extract rgb frames from color_rgb and save as .png;
                rgb_file = os.path.join(folders['rgb_frames'], f"rgb_{frame_count:04d}.png")
                cv2.imwrite(rgb_file, color_bgr)  # RGB format                  color_rgb
                
                
                
                # TO DO based on raw depth (depth) and scaling factor (depth_scale), compute and save depth in meters as .npy              
                
        
            
            # Show preview with alignment info
                cv2.imshow("Recording - RGB", color_bgr)
                cv2.imshow("Recording - Depth", depth_vis)
            
            # Check for quit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("\nUser stopped recording")
                break
            
            frame_count += 1
            
            # Progress
            if frame_count % 30 == 0:
                print(f"Progress: {frame_count}/{max_frames} frames")
    
    finally:
        # Cleanup
        color_writer.release()
        depth_writer.release()
        pipeline.stop()
        cv2.destroyAllWindows()
        
        
# Run recording
if __name__ == "__main__":
    record_video_and_raw()